In [ ]:
# Installing correct production libraries without conflicting decorators
!pip install -q requests gradio langchain-text-splitters langchain-community faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
import requests
from google.colab import userdata

# Safely pulling from Colab Secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
ACTIVE_LLM = "llama-3.1-8b-instant"

def test_groq_connection():
    if not GROQ_API_KEY:
        print("❌ CRITICAL: GROQ_API_KEY not found in Colab Secrets tab.")
        return False

    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
    payload = {
        "model": ACTIVE_LLM,
        "messages": [{"role": "user", "content": "Ping"}],
        "max_tokens": 10
    }
    try:
        res = requests.post(url, headers=headers, json=payload)
        if res.status_code == 200:
            print("✅ Groq API connection successful! Ready for presentation engine.")
            return True
        else:
            print(f"❌ Connection Failed ({res.status_code}): {res.text}")
            return False
    except Exception as e:
        print(f"❌ Network Error: {str(e)}")
        return False

test_groq_connection()

✅ Groq API connection successful! Ready for presentation engine.


True

In [ ]:
# ==========================================
# 3. CORE LOGIC, VECTOR STORAGE & UI LAYOUT
# ==========================================
import os
import requests
import gradio as gr
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
ACTIVE_LLM = "llama-3.1-8b-instant"

# Asal Document Source of Truth
ASAL_RESEARCH_PAPER = """
Towards a Low Cost Multi Parameter Monitoring Framework for Pressure Ulcer Prevention in Resource Limited Healthcare Settings
Team: Neurotech | Project Q2 — 72-Hour Research Hackathon

Section 1: Title & Research Question
Title: Towards a Low Cost Multi Parameter Monitoring Framework for Pressure Ulcer Prevention in Resource Limited Healthcare Settings
Research Question: Can an affordable, sensor integrated, AI driven monitoring system continuously and effectively detect pressure ulcer risk factors in low resource hospital environments where current solutions remain inaccessible or inadequate?

Section 2: Hypothesis
If a low-cost monitoring system integrating pressure, temperature, and moisture sensors with an AI-driven alert mechanism is deployed in resource-limited healthcare settings, then it may improve early detection of pressure ulcer risk factors and reduce preventable ulcer development through continuous monitoring and timely caregiver intervention.

Section 3: Background & Clinical Burden
Pressure ulcers (PUs), also known as bedsores or pressure injuries, are localized injuries to the skin and underlying tissue that develop when sustained pressure restricts blood flow to a region of the body, most commonly over bony prominences such as the sacrum, heels, and hips [1]. The global burden of pressure ulcers is substantial. According to the World Health Organization, PUs affect approximately 1 in 10 hospitalized patients worldwide, with prevalence rates in intensive care units reaching as high as 33% in some settings (WHO, 2023) [3]. In low and middle income countries, the problem is compounded by understaffed wards and high patient-to-nurse ratios where standard 2-hour repositioning protocols are frequently missed. Treating a single full thickness pressure ulcer can cost between $20,000 and $150,000 USD (Bluestein & Javaheri, 2008) [4].

Section 4: Key Findings
Finding 1: Current intelligent monitoring systems focus on posture classification rather than holistic risk prediction. A 2022 systematic review by Silva et al. found that the majority of systems were primarily concerned with detecting lying position using pressure sensor arrays [5].
Finding 2: Multi parameter sensing significantly improves clinical utility. Systems combining pressure data with temperature and moisture sensors provided caregivers with richer, more actionable information. One reviewed study demonstrated that temperature changes at high pressure zones serve as an early biological indicator of tissue ischemia before visible skin damage appears.
Finding 3: Real world deployment exposes major limitations of existing solutions. Several studies reported significant performance degradation when systems were deployed in real ward conditions versus controlled laboratory settings due to bed cushion use, electrical interference, and variable patient body habitus.
Finding 4: AI driven alert systems reduce caregiver workload and improve response times. Automated notifications sent to caregivers' mobile devices resulted in faster repositioning response times compared to standard manual observation protocols.

Section 5: Insight and Proposed Idea (The $20-$45 Architecture)
The proposed framework consists of three components:
1. Low cost sensing layer: Piezoresistive pressure sensors (FSR402 Force Sensitive Resistor ~$5-10), low-resolution temperature sensors (DHT11 or DS18B20 ~$1-3), and capacitive moisture sensors (~$3-5) embedded beneath a standard hospital mattress cover using an ESP32 Development Board (~$3-8), TPU-Coated Medical Fabric (~$2-4), Copper Conductive Thread (~$1-2), and a 5V USB power pack (~$5-10). Total estimated hardware cost is $20–45 USD per bed.
2. Lightweight AI-assisted alert module: Runs on a low-power microcontroller continuously calculating a unified Multi-Parameter Risk Index (RI):
Risk Index (RI) = (W1 * Pressure) + (W2 * Temperature) + (W3 * Moisture)
Based on current pressure ulcer pathophysiology research, weight coefficients are: 50% pressure (W1=0.50), 30% temperature (W2=0.30), and 20% moisture (W3=0.20). Communication operates offline using ESP-NOW or local Wi-Fi mesh networking.
3. Passive longitudinal data logging system: Stores timestamped multimodal measurements, caregiver responses, and intervention outcomes to enable future machine learning model updates.
"""

# Processing RAM Vector Store
print("🧠 Constructing localized neural mapping index...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=40, separators=["\nSection ", "\nFinding ", "\n\n", " "])
docs = text_splitter.create_documents([ASAL_RESEARCH_PAPER])
vector_db = FAISS.from_documents(docs, embeddings)
print("📦 Vector DB operational database mapping ready!")

# Pathophysiology Risk Equation Execution
def run_clinical_math(p, t, m):
    p_factor = float(p) / 100.0
    t_min, t_max = 30.0, 42.0
    t_factor = (float(t) - t_min) / (t_max - t_min)
    t_factor = max(0.0, min(1.0, t_factor))
    m_factor = float(m) / 100.0
    calculated_ri = (0.50 * p_factor) + (0.30 * t_factor) + (0.20 * m_factor)
    return round(calculated_ri, 3)

# RAG Controller Engine
def practical_simulation_engine(pressure, temp, moisture):
    score = run_clinical_math(pressure, temp, moisture)

    if score < 0.42:
        zone_status = "💚 LOW OPERATIONAL RISK"
        clinical_query = "What happens when patients are stable and pressure ulcer risk metrics are low?"
    elif score < 0.70:
        zone_status = "💛 MODERATE CLINICAL ALERT"
        clinical_query = "What biological indicators like temperature change suggest early tissue ischemia before visible damage?"
    else:
        zone_status = "❤️ CRITICAL HIGH RISK EMERGENCY"
        clinical_query = "What hardware layers, sensors, and quick notification alerts are needed when a patient hits critical risk limits?"

    matched_nodes = vector_db.similarity_search(clinical_query, k=2)
    injected_context = "\n---\n".join([n.page_content for n in matched_nodes])

    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}

    system_prompt = (
        "You are an embedded AI Medical System analyzing live bedside telemetry. Your task is to look at the calculated "
        "Risk Index Score and formulate a precise markdown bedside report. You MUST back up your clinical logic "
        "by explicitly mentioning the sensors, costs, findings, or sections found inside the provided research context."
    )

    user_prompt = f"""
    [LIVE HARDWARE NODE DATA]:
    - Dynamic Risk Index (RI): {score}
    - Calculated Status Matrix: {zone_status}
    - Raw Sensors -> Pressure Load: {pressure}%, Core Thermal Node: {temp}°C, Humidity/Moisture: {moisture}%

    [VERIFIED RESEARCH CONTEXT]:
    {injected_context}

    Respond with a clear, markdown-formatted report:
    1. **System Status Overview**: Explain why the current hardware readings triggered this exact Risk Index.
    2. **Verified Paper Reference**: Explicitly link this scenario to specific Findings or Hardware Layers ($20-$45 sensor array) from the paper.
    3. **Actionable Nursing Instructions**: Give precise action steps for hospital staff.
    """

    payload = {
        "model": ACTIVE_LLM,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": 0.2
    }

    try:
        res = requests.post(url, headers=headers, json=payload)
        if res.status_code == 200:
            return res.json()['choices'][0]['message']['content']
        else:
            return f"❌ Groq System Interface Exception ({res.status_code}): {res.text}"
    except Exception as e:
        return f"❌ Network Connection Terminated: {str(e)}"

# Cyber-Luxe Presentation Style Configuration
presentation_css = """
body, .gradio-container {
    background-color: #0a0e17 !important;
    font-family: 'Space Grotesk', system-ui, sans-serif !important;
}
.main-title {
    text-align: center;
    padding: 15px 0;
}
.main-title h1 {
    background: linear-gradient(135deg, #00f2fe 0%, #4facfe 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.3rem !important;
    font-weight: 800 !important;
}
.main-title p {
    color: #94a3b8 !important;
    font-size: 1rem;
}
.glass-container {
    background: rgba(15, 23, 42, 0.65) !important;
    border: 1px solid rgba(0, 242, 254, 0.2) !important;
    border-radius: 12px !important;
    padding: 22px !important;
    box-shadow: 0 8px 32px 0 rgba(0, 0, 0, 0.5) !important;
    backdrop-filter: blur(12px) !important;
}
.action-btn {
    background: linear-gradient(90deg, #00f2fe 0%, #4facfe 100%) !important;
    color: #0a0e17 !important;
    border: none !important;
    font-weight: 700 !important;
    box-shadow: 0 0 15px rgba(0, 242, 254, 0.4) !important;
    cursor: pointer;
    transition: all 0.2s ease-in-out;
}
.action-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 0 25px rgba(0, 242, 254, 0.7) !important;
}
.clinical-output {
    background: rgba(30, 41, 59, 0.4) !important;
    border-left: 4px solid #00f2fe !important;
    padding: 18px !important;
    border-radius: 4px;
}
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="cyan", neutral_hue="slate"), css=presentation_css) as app:
    gr.HTML(
        "<div class='main-title'>"
        "<h1>🏥 Neurotech Practical Bedside RAG Engine</h1>"
        "<p>Colab Live Environment Testing Framework Connected to Groq Cloud API</p>"
        "</div>"
    )

    with gr.Row(elem_classes="glass-container"):
        with gr.Column(scale=2):
            gr.Markdown("### 📡 Bedside Microcontroller Node Simulation")
            p_slider = gr.Slider(0, 100, value=85, label="FSR402 Pressure Matrix Load (%)")
            t_slider = gr.Slider(30.0, 42.0, value=38.2, step=0.1, label="DHT11 Skin Temperature Node (°C)")
            m_slider = gr.Slider(0, 100, value=70, label="Capacitive Moisture Sensor Level (%)")
            run_btn = gr.Button("Transmit Telemetry & Query RAG", elem_classes="action-btn")

        with gr.Column(scale=3):
            gr.Markdown("### 🗲 Live Research-Validated Clinical Response")
            output_markdown = gr.Markdown(elem_classes="clinical-output")

    run_btn.click(
        fn=practical_simulation_engine,
        inputs=[p_slider, t_slider, m_slider],
        outputs=[output_markdown]
    )

app.launch(inline=True, debug=True)

🧠 Constructing localized neural mapping index...


/tmp/ipykernel_2568/2424284762.py:47: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.w

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📦 Vector DB operational database mapping ready!


/tmp/ipykernel_2568/2424284762.py:172: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="cyan", neutral_hue="slate"), css=presentation_css) as app:
/tmp/ipykernel_2568/2424284762.py:172: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="cyan", neutral_hue="slate"), css=presentation_css) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9786e204e1b0247b8d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
